In [1]:
import os
import time
from dotenv import load_dotenv
import requests
from datetime import datetime
import uuid
import io
import zipfile
import pandas as pd
from dateutil.relativedelta import relativedelta
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from loguru import logger
from main import APIRequests

In [2]:
TOKEN_NAME = "КОСТРИК"
load_dotenv()
TOKEN_KEY = os.getenv(TOKEN_NAME)

In [3]:
nomenclature = APIRequests(TOKEN_NAME).get_nomenclature()

2026-04-07 13:25:08.048 | INFO     | main:__init__:37 - Токен КОСТРИК инициализирован
2026-04-07 13:25:08.050 | INFO     | main:get_nomenclature:70 - Запрос URL: https://content-api.wildberries.ru/content/v2/get/cards/list, попытка: 1/5
2026-04-07 13:25:09.394 | INFO     | main:get_nomenclature:98 - Получено карточек: 100
2026-04-07 13:25:09.395 | INFO     | main:get_nomenclature:70 - Запрос URL: https://content-api.wildberries.ru/content/v2/get/cards/list, попытка: 1/5
2026-04-07 13:25:10.269 | INFO     | main:get_nomenclature:98 - Получено карточек: 119


In [4]:
nmIds = [nmId.get('nmID') for nmId in nomenclature]

In [5]:
def chunk_list(nmIds_list: list, chunk_size: int) -> list[list]:
    return [nmIds_list[i:i + chunk_size] for i in range(0, len(nmIds_list), chunk_size)]
nmIds_list = chunk_list(nmIds_list=nmIds, chunk_size=20)

In [6]:
HEADERS = {'Authorization': TOKEN_KEY}
url = 'https://seller-analytics-api.wildberries.ru/api/analytics/v3/sales-funnel/products/history'
report_date =  str(datetime.strptime('04.04.2026', "%d.%m.%Y").strftime("%Y-%m-%d"))

In [ ]:
detail_history_list = []
for nmIds in nmIds_list:
    params = {
        'selectedPeriod': {
        "start": report_date,
        "end": report_date
        },
        'nmIds':nmIds,
        "skipDeletedNm": False,
        "aggregationLevel": "day"
    }
    for attempt in range(5):
        response = requests.post(url=url, headers=HEADERS, json=params)
        if response.status_code == 429:
            if attempt == 5 - 1:
                break
            logger.info(f'{response.status_code} | {response.text} | Ожидание {5*(attempt+1)}сек.')
            time.sleep(5 * (attempt+1))
        else:
            for item in response.json():
                detail_history_list.append(item)
        # else:
        #     for i in range(len(*response.json())):
        #         detail_history_list.append(response.json()[i])
        #     logger.success(f'Данные за {date} из чанка [{nmIds[0]}...{nmIds[-1]}] добавлены')
        #     break
    logger.info(f'Ожидание 20 сек. для избежания ошибки 429!')
    time.sleep(20)
    

all = []
for nmId in detail_history_list:
    item = {}
    item['']

2026-04-07 13:34:20.818 | INFO     | __main__:<module>:17 - 429 | {
    "title": "too many requests",
    "detail": "Limited by global limiter, per seller 36c39ba9-08d1-5a3b-b0f8-d991584edc24; See https://dev.wildberries.ru/openapi/api-information",
    "code": "461a0b83d6bd 2950e93b5fda",
    "requestId": "c8ce9b81342460e78ba22f4dce75be2d",
    "origin": "s2sauth-ca",
    "status": 429,
    "statusText": "Too Many Requests",
    "timestamp": "2026-04-07T07:34:20Z"
}
 | Ожидание 20сек.
2026-04-07 13:34:41.631 | INFO     | __main__:<module>:27 - Ожидание 20 сек. для избежания ошибки 429!
2026-04-07 13:35:03.287 | INFO     | __main__:<module>:17 - 429 | {
    "title": "too many requests",
    "detail": "Limited by global limiter, per seller 36c39ba9-08d1-5a3b-b0f8-d991584edc24; See https://dev.wildberries.ru/openapi/api-information",
    "code": "461a0b83d6bd 2950e93b5fda",
    "requestId": "5a62de66092066351bd639daf40a4d37",
    "origin": "s2sauth-ca",
    "status": 429,
    "statusText

[{'product': {'nmId': 420710677, 'title': 'Туалетная вода мужская Galvanoliori DEEP OCEAN, 100 мл', 'vendorCode': 'GALVANOL000006', 'brandName': 'PARFUMS CONSTANTINE', 'subjectId': 96, 'subjectName': 'Туалетная вода'}, 'history': [{'date': '2026-04-04', 'openCount': 3, 'cartCount': 0, 'orderCount': 0, 'orderSum': 0, 'buyoutCount': 0, 'buyoutSum': 0, 'buyoutPercent': 0, 'addToCartConversion': 0, 'cartToOrderConversion': 0, 'addToWishlistCount': 0}], 'currency': 'RUB'}, {'product': {'nmId': 420755223, 'title': 'Туалетная вода мужская Galvanoliori SPACE BLUE, 100 мл', 'vendorCode': 'GALVANOL000009', 'brandName': 'PARFUMS CONSTANTINE', 'subjectId': 96, 'subjectName': 'Туалетная вода'}, 'history': [{'date': '2026-04-04', 'openCount': 6, 'cartCount': 0, 'orderCount': 0, 'orderSum': 0, 'buyoutCount': 0, 'buyoutSum': 0, 'buyoutPercent': 0, 'addToCartConversion': 0, 'cartToOrderConversion': 0, 'addToWishlistCount': 0}], 'currency': 'RUB'}, {'product': {'nmId': 420762134, 'title': 'Туалетная вод